# **3.3 A Concise Implementation of Linear Regression**
In Section [3.2](./linear_regression.ipynb#321), we only used: (1) tensors for data storage and linear algebra; and (2) automatic differentiation to compute gradients. In fact, since data iterators, loss functions, optimizers, and neural network layers are so commonly used, modern deep learning libraries have already implemented these components for us.

## **3.3.1 Generate the DataSet**

Firstly, generate the dataset.

In [1]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

In [2]:
true_w = torch.tensor([2,-3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

## **3.3.2 Read the DataSet**
We pass the features and labels as parameters to the API and specify the batch_size using the data iterator. Additionally, the boolean value is_train indicates whether we want the data iterator object to shuffle the data in each iteration cycle.

In [3]:
def load_array(data_arrays, batch_size, is_train=True): #@save
    """Construct a PyTorch data iterator"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

In [4]:
batch_size = 10
data_iter = load_array((features, labels), batch_size)

Unlike in Section [3.2](./implementing_linear_regression.ipynb#32), here we use `iter` to create a Python iterator and use `next` to retrieve the first item from the iterator.

In [5]:
next(iter(data_iter))

[tensor([[-1.1352e+00, -1.3553e-01],
         [ 4.4273e-01, -1.7613e+00],
         [-8.6020e-01, -4.7337e-01],
         [ 1.1925e+00, -4.1492e-04],
         [-7.7616e-01,  1.7503e+00],
         [-1.4270e-01,  8.2521e-01],
         [-2.1789e+00,  1.6879e-01],
         [ 1.0025e+00, -1.4553e+00],
         [ 1.8343e-01, -4.0365e-02],
         [-8.1568e-01,  1.1523e+00]]),
 tensor([[ 2.3953],
         [11.0793],
         [ 4.0936],
         [ 6.5856],
         [-3.3276],
         [ 1.1124],
         [-0.7265],
         [11.1699],
         [ 4.7158],
         [-1.3535]])]

## **3.3.3 Define the Models**
For standard deep learning models, we can use the framework’s predefined layers. We first define a model variable `net`, which is an instance of the `Sequential` class. The `Sequential` class chains multiple layers together. When given input data, the `Sequential` instance passes the data to the first layer, then uses the output of the first layer as the input to the second layer, and so on.<br><br>
Looking back at the single-layer network architecture in Figure [3.1.2](./linear_regression.ipynb#fig312), this single layer is called a fully-connected layer because each of its inputs is multiplied by a matrix to produce each of its outputs.<br><br>
In PyTorch, fully connected layers are defined in the `Linear` class. It is worth noting that we pass two arguments to `nn.Linear`. **The first specifies the input feature shape, which is 2, and the second specifies the output feature shape, which is a single scalar, so it is 1.**

In [6]:
# nn is an abbreviation for “neural network”
from torch import nn
net = nn.Sequential(nn.Linear(2, 1))

## **3.3.4 Initialize Model Parameters**
Before using net, we need to initialize the model parameters. Deep learning frameworks typically provide predefined methods for initializing parameters. Here, we specify that each weight parameter should be randomly sampled from a normal distribution with a mean of 0 and a standard deviation of 0.01, while the bias parameters will be initialized to zero.<br><br>
Just as we specified the input and output sizes when constructing `nn.Linear`, we can now directly access the parameters to set their initial values. **We select the first layer in the network using `net[0]`, and then use the `weight.data` and `bias.data` methods to access the parameters. We can also use the `normal_` and `fill_` replacement methods to overwrite the parameter values.**

In [7]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

## **3.3.5 Define the Loss Function**
The MSELoss class is used to calculate the mean squared error, also known as the squared L2 norm. By default, it returns the average of the losses across all samples.

In [8]:
loss = nn.MSELoss()

## **3.3.6 Define an optimization algorithm**
Small-batch stochastic gradient descent is a standard optimization tool for neural networks, and PyTorch implements many variants of this algorithm in its `optim` module. **When we instantiate an SGD instance, we must specify the parameters to be optimized (which can be obtained from our model via `net.parameters()`) and the dictionary of hyperparameters required by the optimization algorithm.** Small-batch stochastic gradient descent only requires setting the `lr` value, which is set to 0.03 here.

In [9]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

## **3.3.7 Training**
To recap: In each iteration cycle, we traverse the entire dataset (train_data) once, continuously drawing a mini-batch of inputs and their corresponding labels from it. For each mini-batch, we perform the following steps:
* Generate predictions by calling net(X) and compute the loss l (forward pass).
* Calculate the gradient by performing backpropagation.
* Update the model parameters by calling the optimizer.<br>

To better evaluate the effectiveness of the training, we calculate the loss after each iteration and print it to monitor the training process.

In [10]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.000223
epoch 2, loss 0.000100
epoch 3, loss 0.000100


Next, we will compare the actual parameters of the generated dataset with the model parameters obtained by training on a limited amount of data. To access the parameters, we first access the desired layer from the network, then read the weights and biases of that layer.

In [11]:
w = net[0].weight.data
print('the evalution loss of w: ', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('the evalution of b: ', true_b - b)

the evalution loss of w:  tensor([-1.0462e-03, -3.8862e-05])
the evalution of b:  tensor([0.0001])


## **Summary**
* We can implement the model more concisely using PyTorch's high-level API.
* In PyTorch, the `data` module provides data processing tools, while the `nn` module defines a wide variety of neural network layers and common loss functions.
* We can initialize parameters by substituting them using methods that end with _.